# 03 Train NAND Best
Best-Path Training (InfoNCE, 818->1024->256).

In [1]:
from pathlib import Path
import sys

RUN_STAGE = "smoke"  # smoke|mini|mid|full
PATHS_CONFIG = "configs/paths.local.yaml"  # alternativ: configs/paths.colab.yaml

def _find_root(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "src").exists() and (c / "configs").exists():
            return c.resolve()
    return start.resolve()

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

RUN_ID = None  # Optional override. Default: auto from notebook 00 latest_run.json
DEVICE = "auto"


ROOT: /home/ubuntu/Author_Name_Disambiguation


In [2]:
import numpy as np
import pandas as pd

from author_name_disambiguation.common.config import (
    load_notebook_context,
    build_run_dirs,
    resolve_run_id,
    write_run_consistency,
)
from author_name_disambiguation.common.io_schema import read_parquet
from author_name_disambiguation.approaches.nand.train import train_nand_across_seeds

CTX = load_notebook_context(run_stage=RUN_STAGE, project_root=ROOT, paths_config=PATHS_CONFIG)
DATA = CTX["DATA"]
ART = CTX["ART"]
MODEL = CTX["MODEL"]
RUN = CTX["RUN"]

latest_context_path = Path(ART["metrics_dir"]) / "latest_run.json"
RUN_ID = resolve_run_id(RUN_ID, latest_context_path, allow_placeholder=False)
RUN_DIRS = build_run_dirs(DATA, ART, RUN_ID)
for key in ["subset_cache", "embeddings", "checkpoints", "metrics"]:
    RUN_DIRS[key].mkdir(parents=True, exist_ok=True)

write_run_consistency(
    run_id=RUN_ID,
    run_stage=RUN_STAGE,
    run_dirs=RUN_DIRS,
    output_path=RUN_DIRS["metrics"] / "03_run_consistency.json",
    extras={"notebook": "03_train_nand_best", "latest_context_path": str(latest_context_path)},
)

training_cfg = MODEL["training"]
subset_dir = RUN_DIRS["subset_cache"]
emb_dir = RUN_DIRS["embeddings"]
ckpt_dir = RUN_DIRS["checkpoints"]
metrics_dir = RUN_DIRS["metrics"]

lspo_mentions = read_parquet(subset_dir / f"lspo_mentions_{RUN_STAGE}.parquet")
lspo_pairs = read_parquet(subset_dir / f"lspo_pairs_{RUN_STAGE}.parquet")
lspo_chars = np.load(emb_dir / f"lspo_chars2vec_{RUN_STAGE}.npy")
lspo_text = np.load(emb_dir / f"lspo_specter_{RUN_STAGE}.npy")

print("RUN_ID:", RUN_ID)
print("Model:", MODEL.get("name"))
print("Loss:", training_cfg.get("loss"))
print("Architecture:", f"{training_cfg['input_dim']} -> {training_cfg['hidden_dim']} -> {training_cfg['output_dim']}")


RUN_ID: smoke_20260213T134837Z_f0fc32b8
Model: nand_best
Loss: infonce
Architecture: 818 -> 1024 -> 256


In [3]:
default_seeds = training_cfg.get("seeds", [1,2,3,4,5])
seeds = [default_seeds[0]] if RUN_STAGE in {"smoke", "mini"} else default_seeds

manifest_path = RUN_DIRS["metrics"] / "03_train_manifest.json"
manifest = train_nand_across_seeds(
    mentions=lspo_mentions,
    pairs=lspo_pairs,
    chars2vec=lspo_chars,
    text_emb=lspo_text,
    model_config=training_cfg,
    seeds=seeds,
    run_id=RUN_ID,
    output_dir=ckpt_dir,
    metrics_output=manifest_path,
    device=DEVICE,
)
manifest


{'run_id': 'smoke_20260213T134837Z_f0fc32b8',
 'runs': [{'seed': 1,
   'checkpoint': '/home/ubuntu/Author_Name_Disambiguation/artifacts/checkpoints/smoke_20260213T134837Z_f0fc32b8/smoke_20260213T134837Z_f0fc32b8_seed1.pt',
   'threshold': -0.039000000000000035,
   'threshold_selection_status': 'ok',
   'threshold_source': 'val_f1_opt',
   'val_class_counts': {'pos': 198, 'neg': 10},
   'test_class_counts': {'pos': 193, 'neg': 11},
   'val_stats': {'f1': 0.9825436408977556,
    'precision': 0.9704433497536946,
    'recall': 0.9949494949494949,
    'accuracy': 0.9663461538461539},
   'test_metrics': {'f1': 0.9846153846153847,
    'precision': 0.9746192893401016,
    'recall': 0.9948186528497409,
    'accuracy': 0.9705882352941176}}],
 'best_seed': 1,
 'best_checkpoint': '/home/ubuntu/Author_Name_Disambiguation/artifacts/checkpoints/smoke_20260213T134837Z_f0fc32b8/smoke_20260213T134837Z_f0fc32b8_seed1.pt',
 'best_threshold': -0.039000000000000035,
 'best_threshold_selection_status': 'ok',

In [4]:
rows = []
for r in manifest["runs"]:
    rows.append({
        "seed": r["seed"],
        "checkpoint": r["checkpoint"],
        "threshold": r["threshold"],
        "val_f1": r["val_stats"].get("f1"),
        "val_precision": r["val_stats"].get("precision"),
        "val_recall": r["val_stats"].get("recall"),
        "test_f1": r["test_metrics"].get("f1"),
        "test_precision": r["test_metrics"].get("precision"),
        "test_recall": r["test_metrics"].get("recall"),
    })

print("Seed summary")
display(pd.DataFrame(rows).sort_values("val_f1", ascending=False))

diag_rows = []
for r in manifest["runs"]:
    v = r.get("val_class_counts", {}) or {}
    t = r.get("test_class_counts", {}) or {}
    diag_rows.append({
        "seed": r["seed"],
        "threshold": r.get("threshold"),
        "threshold_selection_status": r.get("threshold_selection_status"),
        "threshold_source": r.get("threshold_source"),
        "val_pos": int(v.get("pos", 0)),
        "val_neg": int(v.get("neg", 0)),
        "test_pos": int(t.get("pos", 0)),
        "test_neg": int(t.get("neg", 0)),
    })

print("Threshold diagnostics")
display(pd.DataFrame(diag_rows))


Seed summary


,seed,checkpoint,threshold,val_f1,val_precision,val_recall,test_f1,test_precision,test_recall
0,1,/home/ubuntu/Author_Name_Disambiguation/artifa...,-0.039,0.982544,0.970443,0.994949,0.984615,0.974619,0.994819


Threshold diagnostics


,seed,threshold,threshold_selection_status,threshold_source,val_pos,val_neg,test_pos,test_neg
0,1,-0.039,ok,val_f1_opt,198,10,193,11
